Title: demand_prep.ipynb

Purpose: Script to prepare demand data

Author: Onno Nennecke on 04.03.2025 Modified: 24.03.2025

Input data: 

    - This file lies here: 

Output data:

    - This file lies here: 

#### Load packages

In [1]:
import xarray as xr
import numpy as np
# Load cdo package for regridding
from cdo import Cdo
cdo = Cdo()

##### Load population data from vdW Paper

In [2]:
demand_fit_values = xr.open_dataset('/climca/people/onennecke/population_data/vdW_data/demand_fit_values.nc') #('/home/onennecke/EU-renewable-energy-modelling-framework/input_files/demand_fit/demand_fit_values.nc')
demand_population_t2m_grid = xr.open_dataset('/climca/people/onennecke/population_data/vdW_data/population_t2m_grid.nc') #('/home/onennecke/EU-renewable-energy-modelling-framework/input_files/demand_fit/population_t2m_grid.nc')
# demand_population_t2m_grid_weights = xr.open_dataset('/home/onennecke/EU-renewable-energy-modelling-framework/input_files/demand_fit/population_t2m_grid_weights.nc')

##### Create new fit values for the whole week

In [3]:
v_weekday = demand_fit_values.sel(period = 'weekday')
v_weekend = demand_fit_values.sel(period = 'weekend')
v_week = v_weekday * 5/7 + v_weekend * 2/7
v_week.coords['period'] = 'week'
# v_week

In [4]:
demand_fit_values_week = xr.concat([demand_fit_values, v_week.expand_dims('period')], dim='period')
# demand_fit_values_week.sel(country = 9, period = 'week')

# Save new demand fit values
demand_fit_values_week.to_netcdf('/climca/people/onennecke/population_data/demand_fit_values_week.nc')

##### Regrid data

In [5]:
# Define a rough area for Germany
n = 55.5
s = 47
w = 5.5
e = 15.5

demand_population_t2m_grid_de = demand_population_t2m_grid.sel( dict(country = 9, lat=slice(n,s), lon=slice(w,e)))['population']
# demand_population_t2m_grid_weights_de = demand_population_t2m_grid_weights.sel( dict(country = 9, lat=slice(n,s), lon=slice(w,e)))['population']

In [6]:
population_grid_CIESIN = '/climca/people/onennecke/population_data/vdW_data/population_t2m_grid.nc'
# Determine directory for regridding
population_regrid_CIESIN = '/climca/people/onennecke/population_data/population_regrid.nc'

# Take the data from population_t2m_grid and regrid it to the same grid as the capacity data (and save it to population_regrid.nc)
cdo.remapsum(
    '/climca/people/onennecke/Wind_Solar_MaStR/processed_data/wind_offshore_ic_fixed.nc',
    input=population_grid_CIESIN,
    output=population_regrid_CIESIN,
    readCdf=True,
    options="-f nc",
)
'''
For cdo.remapsum() I had to fix the offshore grid with these commands (because the attributes (especially gridtype)
are not saved correctly otherwise:
cdo setgrid,grid.txt wind_offshore_ic_tst.nc wind_offshore_ic_fixed.nc
grid.txt is in the same folder
'''

'\nFor cdo.remapsum() I had to fix the offshore grid with these commands (because the attributes (especially gridtype)\nare not saved correctly otherwise:\ncdo setgrid,grid.txt wind_offshore_ic_tst.nc wind_offshore_ic_fixed.nc\ngrid.txt is in the same folder\n'

In [7]:
population_regrid_CIESIN = xr.open_dataset(population_regrid_CIESIN)
# population_regrid_CIESIN

In [8]:
population_regrid_CIESIN_slice = population_regrid_CIESIN.sel(dict(lat=slice(s,n), lon=slice(w, e), country=9))['population']
# population_regrid_ISIMIP_slice = population_regrid_ISIMIP.sel(dict(lat=slice(s,n), lon=slice(w, e)))['total-population']


In [9]:
population_regrid_CIESIN_slice
# population_regrid_ISIMIP_slice
print(np.nansum(population_regrid_CIESIN_slice.values) / 1000000)
# print(np.nansum(population_regrid_ISIMIP_slice.values) / 1000000)

78.18941650702027


#### Calculate weights

In [10]:
population_regrid_CIESIN_slice_weights = population_regrid_CIESIN_slice / np.nansum(population_regrid_CIESIN_slice.values)

# Save weights
population_regrid_CIESIN_slice_weights.to_netcdf('/climca/people/onennecke/population_data/population_regrid_weights.nc')

# population_regrid_ISIMIP_slice_weights = population_regrid_ISIMIP_slice / np.nansum(population_regrid_ISIMIP_slice.values)